# Processamento de Linguagem Natural e de Imagens
## 2026.2 - Aula 03 - Vetores de Palavras (Word Vectors / Word Embeddings)

Material adaptado e ampliado a partir do curso **Natural Language Processing Demystified**:
- Site: https://nlpdemystified.org
- Repositório: https://github.com/futuremojo/nlp-demystified
- Módulo correspondente: https://www.nlpdemystified.org/course/word-vectors

Versão em português, com explicações adicionais, exemplos comentados e exercícios para os estudantes.

### Objetivos desta aula

Ao final desta aula, o(a) estudante deverá ser capaz de:

1. Entender **o que são vetores de palavras (embeddings)** e por que eles resolvem limitações das representações esparsas (como Bag-of-Words e TF-IDF vistos nas aulas anteriores).
2. Carregar e usar **embeddings estáticos pré-treinados** (Google News / Word2Vec) com a biblioteca **Gensim**.
3. Medir semelhança entre palavras e frases com **similaridade de cosseno**.
4. Explorar propriedades geométricas dos embeddings: palavras semelhantes, o "intruso" de um conjunto, e **analogias por aritmética vetorial** (ex.: Rei - Homem + Mulher = Rainha).
5. Visualizar embeddings de alta dimensão em 2D/3D usando **PCA** e **t-SNE**.
6. Aplicar **embeddings de sentença** (Sentence Transformers) a um conjunto real de *abstracts* científicos para fazer **busca semântica** e **clusterização (K-Means)**.

> **Pré-requisitos**: tokenização, normalização, Bag-of-Words e TF-IDF (aulas anteriores).

### O problema que os embeddings resolvem (intuição)

Nas aulas anteriores representamos textos com **vetores esparsos** (BoW, TF-IDF). Nesse esquema, cada palavra é uma dimensão independente. Isso traz dois problemas:

- **Sem noção de significado**: para o computador, 'gato' e 'felino' são tão diferentes quanto 'gato' e 'avião'. Não há semelhança embutida.
- **Alta dimensionalidade e esparsidade**: o vocabulário pode ter dezenas de milhares de dimensões, quase todas iguais a zero.

Os **word embeddings** representam cada palavra como um vetor **denso** de dimensão fixa (por ex. 300 números reais). Esses vetores são aprendidos de forma que **palavras que aparecem em contextos parecidos fiquem próximas no espaço vetorial** (hipótese distribucional: *"você conhece uma palavra pelas companhias que ela mantém"* — J.R. Firth). Assim, 'gato' e 'felino' ficam próximos, e podemos até fazer contas com significado.

**IMPORTANTE (se usar Google Colab)**

Ative a **aceleração por GPU** em *Runtime > Change Runtime Type*. Em alguns planos o acesso à GPU não é garantido.

Se estiver rodando na nuvem, o notebook pode **expirar** após um período de inatividade. Para rodar o Colab localmente, veja:
https://research.google.com/colaboratory/local-runtimes.html

Este notebook também roda em um ambiente local (Jupyter) com as dependências do arquivo `requirements.txt` desta pasta.

Para demonstrar vetores de palavras usaremos a biblioteca **Gensim**, especializada em modelagem de tópicos e embeddings. O código abaixo importa as bibliotecas que usaremos ao longo do notebook.

- **numpy / pandas**: manipulação de vetores e tabelas.
- **matplotlib**: gráficos.
- **spacy**: processamento linguístico (tokenização etc.).
- **tensorflow / keras**: redes neurais (usado em partes de treino de embeddings).
- **gensim.KeyedVectors**: carregar e consultar vetores pré-treinados.
- **sklearn**: PCA, t-SNE, K-Means e utilidades de aprendizado de máquina.

In [ ]:
import collections
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
import tensorflow as tf

from gensim.models.keyedvectors import KeyedVectors
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

**NOTA**

Neste notebook **não vamos treinar embeddings do zero**. Em vez disso, vamos:
1. Usar embeddings **pré-treinados** (Google News) em uma primeira parte;
2. Aplicar embeddings de sentença a um conjunto real de dados na segunda parte.

Se quiser treinar embeddings do zero, o algoritmo **Skip-Gram com Negative Sampling (SGNS)** é a base do Word2Vec. Na prática, recomenda-se usar a própria Gensim:
- https://radimrehurek.com/gensim/models/word2vec.html
- https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html

---
# Parte 1 - Usando vetores pré-treinados de terceiros

Existem vários pacotes de vetores estáticos pré-treinados (Word2Vec, GloVe, FastText). Aqui usaremos os vetores **Google News**: uma coleção de **3 milhões** de palavras, cada uma representada por um vetor de **300 dimensões**, treinada em cerca de **3 bilhões** de palavras de um corpus de notícias do Google (por volta de 2015).

> **Por que "estáticos"?** Cada palavra tem **um único** vetor, independentemente do contexto. Assim, 'banco' (de sentar) e 'banco' (instituição financeira) compartilham o mesmo vetor. Mais adiante veremos embeddings **contextuais**, que resolvem isso.

Primeiro precisamos baixar o arquivo com os vetores. Ele tem mais de 1 GB, mas cabe no ambiente.

> Se você já tem o arquivo `GoogleNews-vectors-negative300.bin.gz` nesta pasta (ele acompanha o material da aula), pode **pular** a célula de download abaixo e ir direto para a definição de `embedding_file`.

In [ ]:
# Baixa os vetores pre-treinados do Google News a partir do Google Drive.
# Execute apenas se o arquivo .bin.gz ainda nao estiver na pasta.
!pip install gdown
!gdown "https://drive.google.com/uc?id=17Vf3ucDBnHMarELN_yYaRu0lHXdulGTZ"

In [ ]:
# Caminho do arquivo de embeddings. Ajuste se o arquivo estiver em outra pasta.
embedding_file = './GoogleNews-vectors-negative300.bin.gz'

A seguir, a Gensim carrega os vetores pelo módulo **KeyedVectors**, que funciona como um dicionário: a **chave** é a palavra (token) e o **valor** é o vetor.
https://radimrehurek.com/gensim/models/keyedvectors.html

Para economizar tempo e memória, vamos limitar a **200.000** palavras (`limit=200000`). As palavras vêm ordenadas por frequência, então mantemos as mais comuns.

**Parâmetros de `load_word2vec_format`:**
- `binary=True`: o arquivo está no formato binário do Word2Vec.
- `limit=200000`: carrega apenas os N primeiros (mais frequentes) vetores.

> **Consequência do `limit`**: palavras menos comuns não estarão no vocabulário carregado, e consultá-las gerará erro (veremos isso em breve).

In [ ]:
%%time
word_vectors = KeyedVectors.load_word2vec_format(embedding_file, binary=True, limit=200000)

Recuperar o vetor de uma palavra é simplesmente usar o token como chave. O resultado é um vetor NumPy de 300 números reais.

In [ ]:
pizza = word_vectors['pizza']
print(f'Dimensao do vetor: {pizza.shape}')

# O embedding (vetor) da palavra 'pizza'.
print(pizza)

### Similaridade de cosseno (conceito)

Como medir se duas palavras são "parecidas"? Comparando a **direção** de seus vetores. A **similaridade de cosseno** mede o cosseno do ângulo entre dois vetores:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert\, \lVert \mathbf{b} \rVert}$$

- Vale entre **-1 e 1**.
- **Próximo de 1**: vetores apontam na mesma direção (muito semelhantes).
- **Próximo de 0**: vetores ortogonais (sem relação).
- **Próximo de -1**: direções opostas.

Usamos o **ângulo** (e não a distância) porque o que importa é a *direção* semântica, não a magnitude do vetor. O método `similarity` da Gensim calcula isso para duas palavras.
https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similarity

Palavras que esperamos ser relacionadas têm **similaridade mais alta**...

In [ ]:
print(word_vectors.similarity('pizza', 'tomato'))
print(word_vectors.similarity('pizza', 'sauce'))
print(word_vectors.similarity('pizza', 'cheese'))

...em comparação com palavras que **não** esperamos ser relacionadas.

In [ ]:
print(word_vectors.similarity('pizza', 'gorilla'))
print(word_vectors.similarity('pizza', 'tree'))
print(word_vectors.similarity('pizza', 'yoga'))

### Limitação 1: palavras fora do vocabulario (OOV)

Palavras **fora do vocabulario** (*Out-Of-Vocabulary*, OOV) lançam uma exceção. Isso acontece com palavras inventadas, mas também com palavras comuns que ficaram de fora porque carregamos apenas 200.000 vetores.

> Na prática, é preciso **tratar** o caso OOV (por exemplo, ignorar a palavra, usar um vetor de "desconhecido", ou usar modelos como FastText que geram vetores a partir de subpalavras).

In [ ]:
try:
  word_vectors['womblyboo']
except KeyError as e:
  print(e)

### Comparando frases inteiras com `n_similarity`

Podemos comparar duas frases com `n_similarity`, que calcula a similaridade de cosseno entre dois **conjuntos** de palavras.
https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.n_similarity

`n_similarity` espera cada frase como uma **lista de palavras** (por isso usamos `.split()`). Ele calcula a **média** dos vetores de cada frase e compara essas duas médias. Ou seja, o "vetor da frase" é a média dos vetores das palavras.

No exemplo abaixo, 'dog bites man' e 'canine nips human' usam palavras diferentes mas com significado próximo — a similaridade é razoavelmente alta.

In [ ]:
word_vectors.n_similarity("dog bites man".split(), "canine nips human".split())

In [ ]:
word_vectors.n_similarity("martian dolphins are hostile".split(), "i flunked calligraphy school".split())

### Limitação 2: a ordem das palavras se perde

Como usamos a **média** dos vetores, a **ordem** das palavras é ignorada. Duas frases com exatamente as mesmas palavras, mas significados diferentes, recebem similaridade **perfeita (1.0)**.

Este é um ponto importante: a média de embeddings é uma representação simples, mas "cega" à sintaxe. Modelos mais avançados (RNNs, Transformers) levam a ordem em conta.

In [ ]:
word_vectors.n_similarity("dog bites man".split(), "man bites dog".split())

Medidas rapidas de similaridade como esta funcionam melhor quando o corpus e **especifico de um dominio** e a semelhanca depende mais de palavras-chave. Quanto mais especifico, melhor.

Por exemplo, um corpus de manchetes de noticias de negocios tende a funcionar bem.

In [ ]:
s1 = "Volkswagen intends to double electric car sales in China".lower().split()
s2 = "First Toyota with solid state battery will be hybrid".lower().split()
word_vectors.n_similarity(s1, s2)

### Encontrando palavras semelhantes com `most_similar`

O método `most_similar` retorna as palavras cujos vetores estão mais proximos de uma palavra dada.
https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar

**Parametros uteis:**
- `positive`: lista de palavras cujas direcoes queremos **somar**.
- `negative`: lista de palavras cujas direcoes queremos **subtrair** (veremos nas analogias).
- `topn`: quantas palavras retornar.

Cada resultado vem como uma tupla `(palavra, similaridade)`.

In [ ]:
word_vectors.most_similar(positive=['heroes'], topn=10)

Tambem podemos **combinar** vetores primeiro e buscar as palavras mais proximas da media deles. Abaixo somamos os vetores de 'cell' e 'phone' e recuperamos as palavras mais proximas do resultado (note como aparece 'cell_phone', 'cellular', 'telephone'...).

In [ ]:
word_vectors.most_similar(positive=['cell', 'phone'], topn=10)

### Encontrando o "intruso" com `doesnt_match`

Dado um conjunto de palavras, o metodo `doesnt_match` retorna a palavra que "nao combina" com as demais (aquela cujo vetor esta mais distante da media de todos os outros).
https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.doesnt_match

In [ ]:
word_vectors.doesnt_match(["apple", "orange", "hamburger", "banana", "kiwi"])

Vemos o poder do contexto neste exemplo: 'Toyota' e corretamente identificada como a diferente (as outras sao empresas de tecnologia/servicos digitais).

In [ ]:
word_vectors.doesnt_match(["Microsoft", "Apple", "Toyota", "Amazon", "Netflix", "Google"])

### Visualizando vetores com PCA

Nossos vetores tem **300 dimensoes**, impossiveis de desenhar diretamente. Para visualizar, reduzimos para **2 dimensoes** com **PCA (Analise de Componentes Principais)**.
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

**Intuicao do PCA**: ele encontra as direcoes de **maior variancia** dos dados e projeta os pontos nessas direcoes. As duas primeiras componentes principais capturam o maximo de informacao possivel em 2D. E uma tecnica **linear** e deterministica.

A funcao abaixo recebe o modelo e uma lista de palavras, monta a matriz de vetores, aplica PCA para 2D e faz o grafico de dispersao.

In [ ]:
def display_pca_scatterplot(model, words):
    # Monta a matriz (n_palavras x 300) com os vetores das palavras.
    word_vectors = np.array([model[w] for w in words])

    # Aplica PCA e mantem apenas as 2 primeiras componentes principais.
    twodim = PCA().fit_transform(word_vectors)[:,:2]

    # Desenha os pontos e escreve o rotulo de cada palavra ao lado.
    plt.figure(figsize=(10,10))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r', s=128)
    for word, (x,y) in zip(words, twodim):
        plt.text(x+0.05, y+0.05, word)

Observe como palavras relacionadas se agrupam: verbos de natacao, animais (gato/cachorro/felino) e meios de transporte (estrada/carro/onibus).

In [ ]:
display_pca_scatterplot(word_vectors, ['swim', 'swimming', 'cat', 'dog', 'feline', 'road', 'car', 'bus'])

Um exemplo com vocabulario de materiais/metalurgia (util para a segunda parte da aula, sobre ligas de alta entropia).

In [ ]:
display_pca_scatterplot(word_vectors, ['alloy', 'metal', 'Young', 'Bulk', 'ceramic', 'atom', 'entropy'])

### Analogias por aritmetica vetorial

Uma das propriedades mais famosas dos embeddings: podemos resolver **analogias** com **soma e subtracao de vetores**. O exemplo classico e:

$$\text{Rei} - \text{Homem} + \text{Mulher} \approx \text{Rainha}$$

A intuicao e que a **diferenca** entre dois vetores captura uma "relacao" (por ex., a relacao 'capital-de-pais', ou 'genero'). Ao aplicar essa mesma diferenca a outra palavra, chegamos ao termo analogo.

Aqui resolvemos: *Roma esta para Italia assim como Londres esta para ___*. Aritmeticamente: `Italia + Londres - Roma`. Em `most_similar`, isso vira `positive=['Italy', 'London']`, `negative=['Rome']`.

In [ ]:
word_vectors.most_similar(positive=['Italy', 'London'], negative=['Rome'], topn=3)

Visualizar ajuda com a intuicao geometrica: o deslocamento de 'Rome' para 'Italy' e parecido com o de 'London' para 'UK'/'Britain'.

In [ ]:
display_pca_scatterplot(word_vectors, ['Rome', 'Italy', 'London', 'Britain', 'UK'])

### Exercicios - Parte 1

1. **Similaridade**: escolha 5 pares de palavras (em ingles) e preveja, antes de rodar, quais terao similaridade alta e baixa. Confirme com `similarity`.
2. **Analogias**: teste outras analogias, por exemplo `positive=['Paris', 'Germany'], negative=['France']`. O que voce espera? E `positive=['king', 'woman'], negative=['man']`?
3. **Intruso**: monte conjuntos de palavras e veja se `doesnt_match` acerta o intruso. Tente um caso ambiguo e discuta o resultado.
4. **Vocabulario limitado**: aumente o `limit` ao carregar os vetores (ex.: 500000) e veja se palavras antes OOV passam a existir. Qual o custo?
5. **Vies (bias)**: embeddings aprendem os viesses dos dados. Teste analogias sensiveis (ex.: profissoes e genero) e discuta as implicacoes eticas.

---
# Parte 2 - Aplicando aos seus proprios *abstracts*

Agora aplicamos embeddings a um conjunto **real**: `savedrecs.csv`, uma exportacao do **Web of Science** com milhares de *abstracts* cientificos sobre **ligas de alta entropia (high-entropy alloys)**.

O objetivo e mostrar um fluxo pratico de PLN aplicado a literatura cientifica: carregar textos, gerar embeddings, fazer **busca semantica** e **agrupar** documentos por tema.

In [ ]:
# Le a exportacao do Web of Science. Ajuste o nome do arquivo se necessario.
dados_total = pd.read_csv('savedrecs.csv')

Selecionamos a coluna `Abstract` e fazemos uma limpeza basica:
- trocamos strings vazias por `NaN`;
- removemos linhas sem abstract (`dropna`);
- reindexamos para ter indices continuos.

> Sempre inspecione os dados antes de modelar: verifique quantos abstracts sobraram e se ha duplicatas.

In [ ]:
abstracts = dados_total['Abstract']
abstracts.replace('', np.nan, inplace=True)
abstracts.dropna(how='any', inplace=True)
abstracts.reset_index(drop=True, inplace=True)
abstracts

In [ ]:
# Salva os abstracts limpos para reuso.
abstracts.to_csv('savedrecs-dropna.csv')

Como aquecimento, comparamos os dois primeiros abstracts com `n_similarity` (media de vetores Word2Vec). Note que, por serem do mesmo dominio (ligas de alta entropia), a similaridade tende a ser alta.

In [ ]:
s1 = abstracts[0].lower().split()
s2 = abstracts[1].lower().split()
word_vectors.n_similarity(s1, s2)

### Embeddings de sentenca (estaticos vs. contextuais)

A media de vetores Word2Vec ignora ordem e contexto. Uma abordagem moderna e usar **Sentence Transformers**: modelos baseados em **Transformers** que produzem **um unico vetor por frase/documento**, levando o contexto em conta.

- **Embeddings estaticos (Word2Vec/GloVe)**: um vetor fixo por palavra, sem contexto.
- **Embeddings contextuais (BERT, Sentence Transformers)**: o vetor de uma palavra/frase depende das palavras ao redor. Resolvem ambiguidade (ex.: 'banco').

Usaremos o modelo **`all-MiniLM-L6-v2`**: rapido, leve e de bom desempenho geral, gerando vetores de **384 dimensoes**. O metodo `encode` transforma uma lista de textos em uma matriz de embeddings.

In [ ]:
# Instala dependencias, se necessario.
#!pip install sentence_transformers
!pip install tf-keras

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carrega um modelo pre-treinado. 'all-MiniLM-L6-v2' e um bom modelo de uso geral.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Cria os embeddings vetoriais para todos os abstracts.
embeddings = model.encode(abstracts)
print("Formato dos embeddings:", embeddings.shape)

### Busca por semelhanca (busca semantica)

Com os embeddings, podemos fazer **busca semantica**: dada uma consulta em linguagem natural, encontramos os abstracts cujo **significado** e mais proximo — mesmo que nao compartilhem as mesmas palavras.

Passos:
1. Geramos o embedding da consulta com o mesmo modelo.
2. Calculamos a **similaridade de cosseno** entre a consulta e todos os abstracts (`cosine_similarity`).
3. Ordenamos por similaridade (`argsort` invertido com `[::-1]`) e mostramos o top-N.

Isto e a base de sistemas de **recuperacao de informacao** e de **RAG** (Retrieval-Augmented Generation).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

input_sentence = "Have a high Young Modulus"
input_embedding = model.encode([input_sentence])
similarity_scores = cosine_similarity(input_embedding, embeddings)
similarity_scores = similarity_scores.flatten()
most_similar_indices = np.argsort(similarity_scores)[::-1]
top_n = 3
print("--- Abstracts mais semelhantes ---")
for i in range(top_n):
    index = most_similar_indices[i]
    score = similarity_scores[index]
    abstract_text = abstracts[index]
    print(f"\nPosicao {i + 1}: Score = {score:.4f}")
    print(f"Abstract: {abstract_text}")

### Clusterizacao dos abstracts com K-Means

Podemos **agrupar** automaticamente os abstracts por tema usando **K-Means**. O algoritmo particiona os pontos em `k` grupos, minimizando a distancia de cada ponto ao **centroide** do seu grupo.

**Parametros:**
- `n_clusters`: numero de grupos `k` (aqui, 4). E preciso escolher esse valor.
- `random_state=0`: garante reprodutibilidade.
- `n_init=10`: roda o algoritmo 10 vezes com inicializacoes diferentes e mantem a melhor (evita minimos locais ruins).

O resultado (`labels_`) diz a qual cluster cada abstract foi atribuido.

In [ ]:
from sklearn.cluster import KMeans

# Escolhemos o numero de clusters, k. Aqui, 4.
num_clusters = 4

# Inicializa e ajusta o modelo K-Means.
kmeans = KMeans(n_clusters=num_clusters, random_state=0, n_init=10)
kmeans.fit(embeddings)

# Rotulo de cluster de cada abstract.
cluster_labels = kmeans.labels_
print("Rotulos de cluster:", cluster_labels)

### Visualizando os clusters com t-SNE

Os embeddings tem 384 dimensoes. Para ver os clusters, reduzimos para 2D com **t-SNE** (*t-distributed Stochastic Neighbor Embedding*).

**PCA vs. t-SNE:**
- **PCA**: linear, preserva a **estrutura global** (grandes variancias). Rapido e deterministico.
- **t-SNE**: nao-linear, preserva a **vizinhanca local** (pontos proximos continuam proximos). Otimo para visualizar agrupamentos, mas e mais lento, estocastico, e as **distancias globais** no grafico nao devem ser interpretadas literalmente.

**Parametros:**
- `n_components=2`: dimensao de saida (2 para grafico 2D).
- `perplexity=30`: equilibra atencao entre estrutura local e global (valores tipicos entre 5 e 50).
- `random_state=0`: reprodutibilidade.

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

# Inicializa e ajusta o t-SNE para 2 dimensoes.
tsne = TSNE(n_components=2, random_state=0, perplexity=30)
reduced_embeddings = tsne.fit_transform(embeddings)

print("Formato dos embeddings reduzidos:", reduced_embeddings.shape)

In [ ]:
import matplotlib.pyplot as plt

# Grafico de dispersao colorindo cada ponto pelo seu cluster.
plt.figure(figsize=(10, 8))
scatter = plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=cluster_labels, cmap='viridis', s=50)

plt.title("Visualizacao dos clusters de abstracts (t-SNE)", fontsize=16)
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")

legend1 = plt.legend(*scatter.legend_elements(), title="Clusters")
plt.gca().add_artist(legend1)

plt.show()

Tambem podemos visualizar em **3D** (t-SNE com 3 componentes). As cores continuam representando os clusters do K-Means.

In [ ]:
from sklearn.manifold import TSNE
import numpy as np

# t-SNE para 3 dimensoes.
tsne = TSNE(n_components=3, random_state=0, perplexity=30)
reduced_embeddings = tsne.fit_transform(embeddings)

print("Formato dos embeddings reduzidos:", reduced_embeddings.shape)

In [ ]:
# --- Grafico 3D ---
print("Criando o grafico 3D...")
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# c colore os pontos pelo rotulo de cluster; s controla o tamanho.
scatter = ax.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], reduced_embeddings[:, 2],
                     c=cluster_labels, cmap='viridis', s=50)

ax.set_title("Clusters de abstracts em 3D (t-SNE)", fontsize=16)
ax.set_xlabel("Componente t-SNE 1")
ax.set_ylabel("Componente t-SNE 2")
ax.set_zlabel("Componente t-SNE 3")

cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Rotulo do cluster', rotation=270, labelpad=15)

# Ajusta o angulo de visao.
ax.view_init(elev=20, azim=45)

plt.show()

### Quantos clusters usar? O metodo do cotovelo (elbow)

Escolhemos `k=4` arbitrariamente. O **metodo do cotovelo** ajuda a escolher um `k` razoavel: rodamos o K-Means para varios valores de `k` e medimos a **inercia / SSD** (soma das distancias quadradas de cada ponto ao seu centroide).

- A inercia **sempre diminui** conforme `k` aumenta.
- Procuramos o **"cotovelo"** do grafico: o ponto a partir do qual aumentar `k` traz pouca melhora. Esse costuma ser um bom `k`.

In [ ]:
ssd = []
K = range(1, 11)
print("\nRodando K-Means para varios numeros de clusters (k)...")
for k in K:
    try:
        # Ajusta o KMeans para o k atual (n_init=10 para robustez).
        kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
        kmeans.fit(embeddings)
        # Guarda a inercia (SSD).
        ssd.append(kmeans.inertia_)
        print(f"  - k={k}: SSD = {kmeans.inertia_:.2f}")
    except Exception as e:
        print(f"Ocorreu um erro para k={k}: {e}")
        ssd.append(np.nan)

In [ ]:
print("\nPlotando a curva do cotovelo...")
plt.figure(figsize=(10, 8))
plt.plot(K, ssd, 'bx-')
plt.xlabel("Numero de clusters (k)")
plt.ylabel("Soma das distancias quadradas (SSD)")
plt.title("Metodo do cotovelo para o k ideal", fontsize=16)
plt.grid(True)
plt.show()

### Exercicios - Parte 2

1. **Busca semantica**: troque `input_sentence` por outras consultas do dominio (ex.: 'corrosion resistance at high temperature', 'additive manufacturing of alloys') e analise os top-3 resultados. Eles fazem sentido?
2. **Escolha de k**: use o grafico do cotovelo para escolher um `k` melhor que 4, refaca o K-Means e o t-SNE, e compare as visualizacoes.
3. **Interpretacao dos clusters**: para cada cluster, imprima 3-5 abstracts e tente **rotular o tema** de cada grupo (ex.: 'revestimentos', 'propriedades mecanicas', 'corrosao').
4. **Estatico vs. contextual**: compare a similaridade entre dois abstracts usando (a) `n_similarity` do Word2Vec e (b) cosseno dos embeddings do Sentence Transformer. As ordens de semelhanca coincidem?
5. **Outro modelo**: troque `all-MiniLM-L6-v2` por outro Sentence Transformer (ex.: `all-mpnet-base-v2`) e verifique se os clusters mudam. Discuta o trade-off velocidade x qualidade.

---
## Resumo da aula

- **Embeddings** representam palavras/textos como vetores densos onde a **proximidade geometrica reflete proximidade semantica**.
- **Similaridade de cosseno** mede semelhanca pela direcao dos vetores.
- Embeddings estaticos (Word2Vec/Google News) permitem `similarity`, `most_similar`, `doesnt_match` e **analogias** por aritmetica vetorial.
- **Limitacoes** dos estaticos: palavras **OOV**, **perda de ordem** ao mediar vetores e **falta de contexto** (uma palavra = um vetor).
- **Sentence Transformers** geram embeddings **contextuais** por frase/documento, viabilizando **busca semantica** e **clusterizacao**.
- **PCA** e **t-SNE** reduzem dimensao para **visualizacao**; **K-Means** + **metodo do cotovelo** ajudam a **agrupar** e escolher o numero de grupos.

Na proxima aula avancaremos para modelos que levam a **ordem** e o **contexto** em conta (RNNs e Transformers).